In [3]:
import numpy as np
import matplotlib.pyplot as plt
import requests, zipfile, io, os
import pandas as pd

In [4]:

domain = "datasets.techmatrix.it/airml"
token = "DI_xeno_2026"

cities = ["sicilia", "trentino", "venezia", "roma", "puglia",
          "napoli", "firenze", "milano", "bergamo", "bologna"]

for city in cities:
    url = f"https://{domain}/listings/{city}.zip?token={token}"
    resp = requests.get(url, stream=True)
    if resp.ok:
        data_dir = os.path.join("./data", city)
        if not os.path.exists(data_dir):
            os.makedirs(data_dir, exist_ok=True)
        zip_path = os.path.join("./data", f"{city}.zip")
        with open(zip_path, "wb") as f:
            for chunk in resp.iter_content(8192):
                if chunk:
                    f.write(chunk)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(path=data_dir)
        os.remove(zip_path)
    else:
        print(f"Failed to download {city}: {resp.status_code}")

In [5]:
dfs = []
for sub in os.listdir("./data"):
    subpath = os.path.join("./data", sub)
    if not os.path.isdir(subpath):
        continue
    csv_path = os.path.join(subpath, "listings.csv")
    if os.path.exists(csv_path):
        dfs.append(pd.read_csv(csv_path))
    else:
        for root, _, files in os.walk(subpath):
            if "listings.csv" in files:
                dfs.append(pd.read_csv(os.path.join(root, "listings.csv")))
                break

if dfs:
    listings = pd.concat(dfs, ignore_index=True)
else:
    listings = pd.DataFrame()

listings.info()

<class 'pandas.DataFrame'>
RangeIndex: 217194 entries, 0 to 217193
Data columns (total 79 columns):
 #   Column                                        Non-Null Count   Dtype  
---  ------                                        --------------   -----  
 0   id                                            217194 non-null  int64  
 1   listing_url                                   217194 non-null  str    
 2   scrape_id                                     217194 non-null  int64  
 3   last_scraped                                  217194 non-null  str    
 4   source                                        217194 non-null  str    
 5   name                                          217194 non-null  str    
 6   description                                   211756 non-null  str    
 7   neighborhood_overview                         90672 non-null   str    
 8   picture_url                                   217194 non-null  str    
 9   host_id                                       217194 non-nu